# PyANNOW — Step 9b Audio  *(v1.0.0)*

**Teaching a C. elegans worm to approximate Chopin — Step 9b on the full piece.**

This notebook is a focused extraction of Step 9b from
`03_pyannow_naml_progression.ipynb`, applied directly to the **full 229 s
Chopin Nocturne** (no 10 s training-window limitation).  It also demonstrates
the new **v2 piano synthesiser** (ISSUE-003: 3 detuned strings + hammer
transient + room reverb) that replaces the single-string modal model.

## What Step 9b does

| Feature set | Dimension |
|---|---|
| Fourier time embeddings (phase + 12 harmonics + BPM beat) | 27-D |
| Worm PCA scores from 302-neuron activity | k-D |
| ODE residual `q̈ + 2γq̇ + ω²q` (physics feature) | k-D |

An `MLPClassifier(128→64)` is trained on these features against binary Chopin
onset labels, then `calibrated_onset_detect` converts the probabilities into
note events.

## Runtime

Approximately **5–10 minutes** end-to-end (229 s worm simulation + MLP training
+ audio synthesis at 8 kHz).

## References

- Full-progression notebook: `03_pyannow_naml_progression.ipynb`
- Piano synth source: `src/pyannow/composer/piano_synth.py`
- MIDI target: `shared/examples/chopin_nocturne_op_posth_csharp_minor.mid`
- Original WAV: `shared/examples/chopin_nocturne_op_posth_csharp_minor.wav`

## 0. Setup

In [ ]:
import sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from pathlib import Path
from IPython.display import Audio, display, Markdown

from pyannow.targets.midi_target import (
    parse_midi, note_onsets, onset_loss,
    musical_f1, ioi_similarity, calibrated_onset_detect)
from pyannow.composer.piano_synth import synthesise_melody
from pyannow.composer.worm_optimizer_fast import (
    run_forward_fast, MUSCLE_PITCHES_96)
from pyannow.composer.worm_optimizer import generate_neural_activity_302
from pyannow.ion_channels.celegans_hh import DEFAULT_PARAMS
from pyannow.step1_svd.encoder import rsvd, neural_scores, choose_k_by_variance

plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': 0.3})

def _find_repo_root(start=None):
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / 'PyANNOW').is_dir() and (candidate / 'shared').is_dir():
            return candidate
    import pyannow as _pw
    return Path(_pw.__file__).resolve().parent.parent.parent.parent

REPO_ROOT = _find_repo_root()
OUTDIR    = REPO_ROOT / 'PyANNOW' / 'notebooks' / 'step_outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
MIDI_PATH = REPO_ROOT / 'shared' / 'examples' / 'chopin_nocturne_op_posth_csharp_minor.mid'
WAV_PATH  = REPO_ROOT / 'shared' / 'examples' / 'chopin_nocturne_op_posth_csharp_minor.wav'

print('PyANNOW v1.0.0 loaded ✓')
print('REPO_ROOT =', REPO_ROOT)
print('OUTDIR    =', OUTDIR)

## 1. Data — Full Chopin piece (229 s)

Run the 96-cell Boyle forward model for the **complete** 229 s duration,
project onto the top-k RSVD components, and build binary onset labels for all
1 000 Chopin notes.  No training-window cap — the model sees the full piece
from the start (same principle as `04_chopin_score_net.ipynb`).

In [ ]:
t0_setup = time.time()

# ── Full Chopin MIDI ──────────────────────────────────────────────────────────
events_chopin, bpm = parse_midi(MIDI_PATH)
t_on_all  = note_onsets(events_chopin, clip_s=10000)   # all 1 000 notes
T_full_s  = float(t_on_all[-1] + 2.0)                 # ≈ 229 s
print(f'Chopin: {len(events_chopin)} notes, {bpm:.0f} BPM, '
      f'{t_on_all[-1]:.1f} s ({t_on_all[-1]/60:.1f} min)')

# ── 96-cell Boyle forward model (full piece) ──────────────────────────────────
result_full = run_forward_fast(
    DEFAULT_PARAMS, duration_s=T_full_s, dt_ms=0.5,
    drive_freq_hz=1.5, drive_amplitude=12.0, random_seed=42)
t_full     = result_full['t_arr_ms'] * 1e-3   # (T_full_pts,)  seconds
T_full_pts = len(t_full)
dt_full    = float(t_full[1] - t_full[0])
print(f'Worm simulation: T={T_full_pts} pts, {T_full_s:.0f} s  '
      f'({time.time()-t0_setup:.1f} s elapsed)')

# ── 302-neuron activity + RSVD (Step 1a) ─────────────────────────────────────
X_neural = generate_neural_activity_302(
    n_steps=T_full_pts, dt_ms=0.5, drive_freq_hz=1.5, seed=42)
k_worm   = choose_k_by_variance(X_neural, variance_threshold=0.90)
k_worm   = min(k_worm, 4)
U_k, s_k, Vt_k = rsvd(X_neural, k=k_worm, q=1, seed=0)
Z_worm   = neural_scores(X_neural, U_k).T              # (T_full_pts, k_worm)
print(f'RSVD: k={k_worm}  Z_worm={Z_worm.shape}  '
      f'({time.time()-t0_setup:.1f} s elapsed)')

# ── Binary onset labels (±50 ms window) ──────────────────────────────────────
y_onset   = np.zeros(T_full_pts, dtype=int)
tol_steps = int(0.05 / dt_full)
for t_on in t_on_all:
    idx = np.searchsorted(t_full, t_on)
    lo, hi = max(0, idx - tol_steps), min(T_full_pts, idx + tol_steps + 1)
    y_onset[lo:hi] = 1
print(f'Onset labels: {y_onset.sum()} positive / {T_full_pts} total '
      f'({y_onset.mean()*100:.2f}% positive)')
print(f'Setup total: {time.time()-t0_setup:.1f} s')

## 2. Step 9b — Physics-residual enhanced MLP (full piece)

**Feature set:**
- **Fourier time embeddings** (27-D): global phase + 12 sin/cos harmonics
  + BPM beat-phase.  Phase is normalised over the full 229 s so the model never
  extrapolates beyond the training range.
- **Worm PCA scores** (k-D): 302-D neural activity projected onto the top-k
  RSVD components — carries the biological locomotion signal.
- **ODE residual** (k-D): `r(t) = q̈ + 2γq̇ + ω²q` (damped oscillator
  residual).  Spikes at muscle contraction → relaxation transitions (onsets).
  Adding it as a feature tests whether the physics structure is redundant given
  `Z_worm`.

**Verdict (v0.9.0 — 10 s window):** the ODE residual ties Step 9 exactly
(F1 ≈ 0.879) — it is a linear function of `Z_worm` and its numerical
derivatives, which the MLP can already derive implicitly.  On the **full piece**
the combined feature set benefits from no extrapolation gap.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

t0_9b = time.time()

# ── Fourier time embeddings (Step 9 prerequisite) ────────────────────────────
# Phase normalised over full 229 s — no extrapolation beyond training range
phase_s  = t_full / T_full_s                           # [0, 1]
n_h9     = 12                                          # harmonics
t_feats  = [phase_s[:, None]]
for k_h in range(1, n_h9 + 1):
    ang = 2.0 * np.pi * k_h * phase_s
    t_feats += [np.sin(ang)[:, None], np.cos(ang)[:, None]]
beat_ph  = t_full * bpm / 60.0
t_feats += [np.sin(2 * np.pi * beat_ph)[:, None],
            np.cos(2 * np.pi * beat_ph)[:, None]]
T_feats  = np.concatenate(t_feats, axis=1)             # (T, 27)

# ── Standardized worm PCA scores ─────────────────────────────────────────────
scaler   = StandardScaler().fit(Z_worm)
Zs       = scaler.transform(Z_worm)                    # (T, k_worm)

# ── ODE residual: q̈ + 2γq̇ + ω²q ────────────────────────────────────────────
dZw      = np.gradient(Z_worm, dt_full, axis=0)
d2Zw     = np.gradient(dZw,    dt_full, axis=0)
ode_res  = d2Zw + 2.0 * 0.3 * dZw + 2.5**2 * Z_worm
ode_res_s = StandardScaler().fit_transform(ode_res)
print(f'ODE residual: shape={ode_res.shape}  |r|={np.abs(ode_res).mean():.4f}')

# ── Combined feature matrix ───────────────────────────────────────────────────
X9b = np.hstack([T_feats, Zs, ode_res_s])             # (T, 27+2*k_worm)
print(f'Feature matrix: {X9b.shape}  '
      f'(Fourier={T_feats.shape[1]}-D + worm={k_worm}-D + ODE={k_worm}-D)')

# ── Subsample + class-balance sample weights ─────────────────────────────────
rng    = np.random.default_rng(0)
sub    = min(30000, T_full_pts)
idx_s  = rng.choice(T_full_pts, sub, replace=False)
X_s, y_s = X9b[idx_s], y_onset[idx_s]
pos_r  = y_s.sum() / max(1, len(y_s) - y_s.sum())
sw     = np.where(y_s == 1, 1.0 / max(pos_r, 1e-6), 1.0)

# ── MLP training ──────────────────────────────────────────────────────────────
mlp9b  = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation='relu',
    solver='adam', max_iter=300, early_stopping=True,
    n_iter_no_change=15, random_state=0, verbose=False)
mlp9b.fit(X_s, y_s, sample_weight=sw)
print(f'MLP converged in {mlp9b.n_iter_} iterations  '
      f'({time.time()-t0_9b:.1f} s elapsed)')

# ── Onset detection ───────────────────────────────────────────────────────────
probs9b  = mlp9b.predict_proba(X9b)[:, 1]
onsets9b = calibrated_onset_detect(
    probs9b, t_on_all, t_full, tol_s=0.05, refractory_s=0.28)
F1_9b    = musical_f1(onsets9b, t_on_all, window_s=T_full_s)
L9b      = onset_loss(onsets9b, t_on_all, window_s=T_full_s)
I9b      = ioi_similarity(onsets9b, t_on_all, window_s=T_full_s)

print(f'\nStep 9b (full {T_full_s:.0f} s): {len(onsets9b)} onsets detected')
print(f'  F1 = {F1_9b["f1"]:.6f}')
print(f'  onset_loss = {L9b:.5f}')
print(f'  IOI similarity = {I9b:.3f}')
print(f'  Total elapsed: {time.time()-t0_9b:.1f} s')

# Probability trace — first 10 s
fig, ax = plt.subplots(figsize=(12, 3))
mask = t_full <= 10.0
ax.plot(t_full[mask], probs9b[mask], lw=0.8, color='steelblue')
for t_on in t_on_all[t_on_all <= 10.0]:
    ax.axvline(t_on, color='red', alpha=0.4, lw=0.8)
ax.set(xlabel='time (s)', ylabel='onset probability',
       title=f'Step 9b onset probability — first 10 s  (F1={F1_9b["f1"]:.4f})')
plt.tight_layout()
plt.savefig(OUTDIR / 'step9b_v1_probability.png', dpi=100)
plt.show()

## 3. Audio synthesis — v2 piano (ISSUE-003)

The new `render_string_v2` synthesiser (v1.0.0) replaces the single-string
modal model with:
- **3 detuned strings** at 0, +5, −5 cents — unison choir gives ≈ 3 Hz beating
  at A4, eliminating the "tin can" resonance.
- **Hammer-impact transient** — 4 ms shaped white-noise burst models the
  physical hammer-string collision.
- **Room reverb** — 20% wet synthetic impulse response (RT60 ≈ 0.3 s).

Output sample rate: **8 000 Hz** → WAV file ≈ **3.5 MB** (same as the original
recorded WAV).

In [ ]:
FS_OUT = 8000   # matches source WAV: 8 kHz → ~3.5 MB for 229 s
t0_synth = time.time()

# ── Map detected onsets → pitch events ───────────────────────────────────────
_pm = result_full['pitch_map']
_nm = len(_pm)
events_worm = [(_t, _pm[_ki % _nm], 64) for _ki, _t in enumerate(onsets9b)]

# ── Synthesise: v2 piano (3 strings + hammer + reverb) ───────────────────────
print('Synthesising worm piano (v2, 3 strings + reverb)...')
audio_worm, _ = synthesise_melody(
    events_worm, duration_s=T_full_s, fs=FS_OUT,
    use_v2=True, reverb=True, reverb_mix=0.20)

print('Synthesising Chopin MIDI → v2 piano...')
audio_chop, _ = synthesise_melody(
    events_chopin, duration_s=T_full_s, fs=FS_OUT,
    use_v2=True, reverb=True, reverb_mix=0.20)

print(f'Synthesis: {time.time()-t0_synth:.1f} s')

# ── Save WAVs ─────────────────────────────────────────────────────────────────
wav_worm = OUTDIR / 'worm_step9b_v1.wav'
wav_chop = OUTDIR / 'chopin_synth_v1.wav'
wavfile.write(str(wav_worm), FS_OUT, (audio_worm * 28000).astype('int16'))
wavfile.write(str(wav_chop), FS_OUT, (audio_chop * 28000).astype('int16'))

import os
sz_w = os.path.getsize(wav_worm) / 1e6
sz_c = os.path.getsize(wav_chop) / 1e6
print(f'Saved: {wav_worm.name}  ({sz_w:.2f} MB,  {T_full_s:.0f} s)')
print(f'Saved: {wav_chop.name}  ({sz_c:.2f} MB,  {T_full_s:.0f} s)')

# ── Waveform comparison (first 10 s) ─────────────────────────────────────────
t_plot = np.arange(int(10 * FS_OUT)) / FS_OUT
fig, axes = plt.subplots(2, 1, figsize=(12, 4))
axes[0].plot(t_plot, audio_worm[:len(t_plot)], lw=0.5, color='steelblue')
axes[0].set(title=f'Worm Step 9b (v1.0.0) — first 10 s  F1={F1_9b["f1"]:.3f}',
            xlabel='time (s)', ylabel='amplitude')
axes[1].plot(t_plot, audio_chop[:len(t_plot)], lw=0.5, color='crimson')
axes[1].set(title='Chopin MIDI → v2 piano — first 10 s',
            xlabel='time (s)', ylabel='amplitude')
plt.tight_layout()
plt.savefig(OUTDIR / 'step9b_v1_waveform.png', dpi=100)
plt.show()

print('\nWorm piano (Step 9b, v2 synth):')
display(Audio(audio_worm, rate=FS_OUT, autoplay=False))
print('Chopin MIDI → v2 piano:')
display(Audio(audio_chop, rate=FS_OUT, autoplay=False))

## 4. Original Chopin — real piano recording

The recorded WAV (`chopin_nocturne_op_posth_csharp_minor.wav`, 3.6 MB, 8 kHz)
is the acoustic reference against which the synthesised audio is compared.

In [ ]:
fs_wav, audio_orig = wavfile.read(str(WAV_PATH))
if audio_orig.dtype == np.int16:
    audio_orig = audio_orig.astype(np.float32) / 32768.0
if audio_orig.ndim == 2:
    audio_orig = audio_orig.mean(axis=1)
dur_orig = len(audio_orig) / fs_wav

import os
sz_orig = os.path.getsize(WAV_PATH) / 1e6
print(f'Original Chopin WAV: {dur_orig:.1f} s / {dur_orig/60:.1f} min  '
      f'fs={fs_wav} Hz  {sz_orig:.1f} MB')
print(f'Synthesised worm WAV: {sz_w:.2f} MB  (target: ≤ 4 MB ✓)')
display(Audio(audio_orig, rate=fs_wav, autoplay=False))

## Summary  *(v1.0.0)*

| Item | Value |
|---|---|
| Step | 9b (Worm + Fourier time + ODE residual) |
| Training data | Full 229 s Chopin piece (no window cap) |
| Feature dimension | 27 + k + k = 27 + 2k |
| Piano synthesiser | `render_string_v2` — 3 strings + hammer noise + reverb |
| Output sample rate | 8 000 Hz |
| Output WAV size | ≈ 3.5 MB (source: 3.6 MB) |
| ISSUE-003 | Resolved — v2 piano replaces tin-can modal synth |
| ISSUE-004 | Resolved — `duration_s=None` renders full piece by default |

### v1.0.0 changes

- **`render_string_v2`** in `piano_synth.py` — 3 detuned strings (0/+5/−5 cents)
  eliminate the single-mode tin-can resonance; 4 ms hammer-impact burst adds
  percussive attack.  Fixes ISSUE-003.
- **Room reverb** post-process in `synthesise_melody` via
  `scipy.signal.fftconvolve` (20% wet, RT60 ≈ 0.3 s).  `use_v2=True` and
  `reverb=True` are now the defaults.
- **`duration_s=None`** default in `synthesise_melody` — derives output length
  from the last event time so the full Chopin piece renders without truncation.
  Fixes ISSUE-004.
- **This notebook** (`05_pyannow_step9b_audio.ipynb`) — Step 9b extracted from
  the full progression notebook and applied directly to the 229 s piece.